# GeoCadastra: resumable GPU training
Select Runtime → Change runtime type → GPU. This trains the actual RGB + height MultiTaskNet on **synthetic** wards. It does not train on the downloaded New Zealand PNGs. Those need matched parcel supervision and preparation first.
Upload `stage4_gpu_bundle.zip` from the accompanying artifacts. Keep Colab’s installed CUDA PyTorch. Drive is mounted before training. GPU availability varies; 400 epochs does not guarantee the accuracy target.


In [ ]:
from google.colab import files, drive
from pathlib import Path
import zipfile, subprocess, sys
uploaded = files.upload()  # Select stage4_gpu_bundle.zip
archive = Path("stage4_gpu_bundle.zip")
assert archive.exists(), "Upload stage4_gpu_bundle.zip"
ROOT = Path("/content/SIH_2026")
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert (ROOT / name).resolve().is_relative_to(ROOT.resolve()), "Unsafe archive path"
    z.extractall(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "scripts/colab/requirements.txt")], check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime, then rerun"
print(torch.__version__, torch.cuda.get_device_name(0))
drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/GeoCadastra/stage4_64w_400e_b16_v16')
RUN.mkdir(parents=True, exist_ok=True)
base = [sys.executable, '-u', str(ROOT / 'scripts/colab/train_stage4_gpu.py'),
        '--out', str(RUN), '--wards', '64', '--val-wards', '16',
        '--batch-size', '16', '--epochs', '400']


Benchmark one epoch first. Check the printed run time and validation loss. The first chunk includes dataset construction; this is not a precise full-run estimate. Checkpoints are saved every epoch. Rerunning resumes the last completed epoch with the same schedule.

In [ ]:
subprocess.run(base + ['--epochs-this-run', '1'] +
               (['--resume'] if (RUN / 'last.pt').exists() else []), check=True)


Continue to 400 total epochs. Each chunk prints progress. After a disconnect, upload the same bundle, mount Drive and rerun. Do not change wards, batch size or total epochs when resuming; use a new output folder for a different experiment.

In [ ]:
import json
while True:
    saved = torch.load(RUN / 'last.pt', map_location='cpu', weights_only=True)
    completed = saved['epoch']
    del saved
    if completed >= 400:
        break
    subprocess.run(base + ['--epochs-this-run', '10', '--resume'], check=True)
    print(json.loads((RUN / 'progress.json').read_text()))


Evaluate **last.pt.best**, selected by validation loss, on ten separate synthetic test wards. This loads the saved weights without retraining. Inspect precision and reverse distance too: a one-sided boundary distance alone can reward excessive predicted boundaries. Synthetic success does not establish cadastral accuracy or calibration.

In [ ]:
subprocess.run(base + ['--evaluate'], check=True)
print('Keep:', RUN / 'last.pt.best', RUN / 'heldout_metrics.json')


`last.pt` resumes training; `last.pt.best` is the selected model. Do not use the Stage 4 pytest command to evaluate these files: that test trains its own model. For deployment, retain an immutable copy of selected weights and its evaluation report; do not automatically promote unverified weights.